# 列簡化階梯形 (RREF) 與秩 (Rank)

本筆記本使用 **SymPy** 介紹線性代數中兩個互相緊密相關的概念：

1. **列運算與高斯消去** — 解方程組的基本工具
2. **列階梯形 (REF)** 與 **列簡化階梯形 (RREF)** — 矩陣的標準形
3. **主元 (Pivot)** — RREF 中的關鍵位置
4. **秩 (Rank)** — 矩陣「獨立資訊量」的度量
5. **用 RREF 解線性方程組** — 唯一解、無限多解、無解

In [1]:
import sympy as sp
from IPython.display import display, Math

def display_large(expr):
    """Display a sympy expression in LaTeX with large font size."""
    display(Math(r"\large{%s}" % sp.latex(expr)))


def display_huge(expr):
    """Display a sympy expression in LaTeX with huge font size."""
    display(Math(r"\huge{%s}" % sp.latex(expr)))

## 1. 基本列運算 (Elementary Row Operations)

解線性方程組時，我們對**增廣矩陣**做三種允許的列運算，解不變：

| 運算 | 說明 | SymPy |
|------|------|-------|
| 交換兩列 | $R_i \leftrightarrow R_j$ | `A.row_swap(i, j)` |
| 某列乘非零常數 | $R_i \leftarrow c\, R_i$ | `A.row_op(i, lambda v, j: c*v)` |
| 某列加上另一列的倍數 | $R_i \leftarrow R_i + c\, R_j$ | `A.row_op(i, lambda v, j: v + c*A[j, :])` |

這些運算合稱為**高斯消去法 (Gaussian elimination)** 的基礎。

In [ ]:
# 示範：基本列運算（Elementary Row Operations）的每一步，並以中文詳細解釋

# 1. 建立一個 3x3 的矩陣 A
# 每一列視為一組線性方程式的係數
A = sp.Matrix([
    [1, 2, 3],    # 第一列：x + 2y + 3z
    [2, 5, 7],    # 第二列：2x + 5y + 7z
    [1, 1, 2],    # 第三列：x + y + 2z
])

print("原始矩陣 A：")
# 用自訂的 display_huge 以特大字體顯示矩陣 A，方便檢視
display_huge(A)

# 2. 第一個列運算：將第 2 列減去第 1 列的 2 倍
# R2 ← R2 - 2*R1
# 詳細中文說明：
# 這一步的目的是將第 2 列的第 1 欄（主元下方）歸零，方便往階梯狀前進
# 注意：A_op = A.copy()，是為了保留原始的 A，不影響後續操作
A_op = A.copy()
A_op.row_op(
    1,                         # 目標為第 2 列（Python 的 index 從 0 開始，因此 1 代表第 2 列）
    lambda v, j: v - 2 * A_op[0, j]   # 對第 2 列的每個元素 v，減去第 1 列對應元素乘以 2
    # A_op[0, j] 代表矩陣 A_op 的第 1 列（Python 的 index 從 0 開始）的第 j 欄元素，
    # 也就是主元列中對應欄位的值。在 lambda v, j 中的 v 是目標列（這裡是第 2 列）的元素，
    # A_op[0, j] 則是第 1 列相同位置的元素。例如 j=0 則 A_op[0,0]=1，j=1 則 A_op[0,1]=2。
)
print("列運算後：R2 ← R2 - 2*R1")
display_huge(A_op)

# 3. 第二個列運算：將第 3 列減去第 1 列
# R3 ← R3 - R1
# 詳細說明：
# 這一步讓第 3 列的第 1 欄也成為 0，這是高斯消去法的基本步驟
A_op.row_op(
    2,                         # 目標為第 3 列
    lambda v, j: v - A_op[0, j]   # 對第 3 列的每個元素 v，減去第 1 列對應元素
)
print("列運算後：R3 ← R3 - R1")
display_huge(A_op)

Original A:


<IPython.core.display.Math object>

After R2 <- R2 - 2*R1:


<IPython.core.display.Math object>

After R3 <- R3 - R1:


<IPython.core.display.Math object>

## 2. 列階梯形 (REF) 與列簡化階梯形 (RREF)

### 2.1 列階梯形 (Row Echelon Form, REF)

矩陣為 **REF** 若滿足：

1. 所有全零列都在底部
2. 每一列第一個非零元素（**主元, pivot**）出現在上一列主元的**右側**
3. 主元下方的元素皆為 $0$（階梯結構）

### 2.2 列簡化階梯形 (Reduced Row Echelon Form, RREF)

在 REF 之上再加兩條：

4. 每個主元等於 **$1$**
5. 主元所在**行（column）**的其他元素皆為 **$0$**（主元上下都清乾淨）

每個矩陣的 RREF **唯一**；REF 則不唯一（主元可為任意非零數）。

$$
\text{例子}\quad
\begin{pmatrix}
1 & 2 & 0 & -1 \\
0 & 0 & 1 & 3 \\
0 & 0 & 0 & 0
\end{pmatrix}
\quad\text{已是 RREF（主元在第 1、3 行）}
$$

## 3. 用 SymPy 計算 RREF

`A.rref()` 回傳一個 tuple：`(RREF 矩陣, 主元所在行的索引 tuple)`。

```python
R, pivots = A.rref()
```

In [3]:
A = sp.Matrix([
    [1, 2, 3],
    [2, 4, 8],
    [3, 6, 10],
])

print("A =")
display_huge(A)

R, pivots = A.rref()

print("RREF(A) =")
display_huge(R)

print("Pivot column indices:", pivots)

A =


<IPython.core.display.Math object>

RREF(A) =


<IPython.core.display.Math object>

Pivot column indices: (0, 2)


In [4]:
# Another example: identity is already in RREF
I = sp.eye(3)
print("I_3 (already RREF):")
display_huge(I)
print("pivots:", I.rref()[1])

# Singular / dependent rows
B = sp.Matrix([
    [1, 2, 3],
    [2, 4, 6],
    [1, 1, 1],
])
print("\nB =")
display_huge(B)
R_B, pivots_B = B.rref()
print("RREF(B) =")
display_huge(R_B)
print("pivots:", pivots_B)

I_3 (already RREF):


<IPython.core.display.Math object>

pivots: (0, 1, 2)

B =


<IPython.core.display.Math object>

RREF(B) =


<IPython.core.display.Math object>

pivots: (0, 1)


## 4. 主元 (Pivot)

- **主元行 (pivot column)**：RREF 中含有主元 $1$ 的那些行
- **自由變數行 (free column)**：沒有主元的行，對應方程組中的自由變數

主元的個數 = 非零列的個數 = 矩陣的**秩**（下一節）。

In [5]:
C = sp.Matrix([
    [1, 3, 1, 9],
    [1, 1, -1, 1],
    [3, 11, 5, 35],
])

print("C =")
display_huge(C)

R_C, pivots_C = C.rref()
print("RREF(C) =")
display_huge(R_C)

n_cols = C.cols
free_cols = [j for j in range(n_cols) if j not in pivots_C]

print("Pivot columns:", list(pivots_C))
print("Free columns:", free_cols)
print("Number of pivots:", len(pivots_C))

C =


<IPython.core.display.Math object>

RREF(C) =


<IPython.core.display.Math object>

Pivot columns: [0, 1]
Free columns: [2, 3]
Number of pivots: 2


## 5. 秩 (Rank)

矩陣 $A$ 的**秩** $\operatorname{rank}(A)$ 有多種等價定義：

1. RREF（或 REF）中**非零列**的個數
2. **主元**的個數
3. 最大線性獨立**列向量**的個數（行秩 = 列秩）
4. 最大線性獨立**行向量**的個數

直覺：秩衡量矩陣裡「真正獨立的資訊量」。

對 $m \times n$ 矩陣：

$$
0 \le \operatorname{rank}(A) \le \min(m, n)
$$

在 SymPy 中：`A.rank()`。

In [6]:
examples = {
    "full rank 2x2": sp.Matrix([[1, 2], [3, 4]]),
    "singular 2x2": sp.Matrix([[1, 2], [2, 4]]),
    "3x3 rank 2": sp.Matrix([[1, 2, 3], [2, 4, 6], [1, 1, 1]]),
    "3x4": sp.Matrix([[1, 3, 1, 9], [1, 1, -1, 1], [3, 11, 5, 35]]),
    "zero matrix": sp.zeros(2, 3),
}

for name, M in examples.items():
    R, pivots = M.rref()
    print(f"--- {name} ---")
    display_large(M)
    print(f"rank = {M.rank()},  #pivots = {len(pivots)},  pivots = {pivots}")
    print()

--- full rank 2x2 ---


<IPython.core.display.Math object>

rank = 2,  #pivots = 2,  pivots = (0, 1)

--- singular 2x2 ---


<IPython.core.display.Math object>

rank = 1,  #pivots = 1,  pivots = (0,)

--- 3x3 rank 2 ---


...

rank = 2,  #pivots = 2,  pivots = (0, 1)

--- 3x4 ---


<IPython.core.display.Math object>

rank = 2,  #pivots = 2,  pivots = (0, 1)

--- zero matrix ---


<IPython.core.display.Math object>

rank = 0,  #pivots = 0,  pivots = ()



### 5.1 秩的重要性質

- $\operatorname{rank}(A) = \operatorname{rank}(A^T)$（行秩 = 列秩）
- $\operatorname{rank}(AB) \le \min\big(\operatorname{rank}(A), \operatorname{rank}(B)\big)$
- 對 $n \times n$ 方陣：
  - $\operatorname{rank}(A) = n$ $\Leftrightarrow$ $A$ **滿秩 (full rank)** $\Leftrightarrow$ $A$ **可逆** $\Leftrightarrow$ $\det(A) \neq 0$
  - $\operatorname{rank}(A) < n$ $\Leftrightarrow$ $A$ **奇異 (singular)**

In [7]:
P = sp.Matrix([[1, 2], [3, 4]])
Q = sp.Matrix([[1, 2], [2, 4]])  # singular

print("P =")
display_huge(P)
print(f"rank(P) = {P.rank()}, det(P) = {P.det()}, invertible? {P.det() != 0}")

print("\nQ =")
display_huge(Q)
print(f"rank(Q) = {Q.rank()}, det(Q) = {Q.det()}, invertible? {Q.det() != 0}")

print("\nrank(P) == rank(P.T) ?", P.rank() == P.T.rank())
print("rank(PQ) <= min(rank(P), rank(Q)) ?",
      (P * Q).rank() <= min(P.rank(), Q.rank()))

P =


<IPython.core.display.Math object>

rank(P) = 2, det(P) = -2, invertible? True

Q =


<IPython.core.display.Math object>

rank(Q) = 1, det(Q) = 0, invertible? False

rank(P) == rank(P.T) ? True
rank(PQ) <= min(rank(P), rank(Q)) ? True


## 6. 用 RREF 解線性方程組 $A\mathbf{x} = \mathbf{b}$

步驟：

1. 組成**增廣矩陣** $[A \mid \mathbf{b}]$
2. 化成 RREF
3. 依主元與最後一行判斷解的情形

| 情形 | 條件 | 結果 |
|------|------|------|
| 無解 | 出現 $[0\;\cdots\;0 \mid c]$ 且 $c \neq 0$ | 矛盾 |
| 唯一解 | 每個變數都有主元（無自由變數） | 一個解 |
| 無限多解 | 有自由變數，且無矛盾列 | 參數解 |

等價說法（係數矩陣 vs 增廣矩陣）：

- $\operatorname{rank}(A) < \operatorname{rank}([A\mid b])$ → **無解**
- $\operatorname{rank}(A) = \operatorname{rank}([A\mid b]) = n$（變數個數）→ **唯一解**
- $\operatorname{rank}(A) = \operatorname{rank}([A\mid b]) < n$ → **無限多解**

### 6.1 唯一解

$$
\begin{cases}
x + 2y = 5 \\
3x + 4y = 11
\end{cases}
$$

In [8]:
A1 = sp.Matrix([[1, 2], [3, 4]])
b1 = sp.Matrix([5, 11])
aug1 = A1.row_join(b1)

print("Augmented [A|b] =")
display_huge(aug1)

R1, pivots1 = aug1.rref()
print("RREF =")
display_huge(R1)

print(f"rank(A) = {A1.rank()}, rank([A|b]) = {aug1.rank()}, n = {A1.cols}")
print("Solution via A.solve(b):")
display_huge(A1.solve(b1))

Augmented [A|b] =


<IPython.core.display.Math object>

RREF =


<IPython.core.display.Math object>

rank(A) = 2, rank([A|b]) = 2, n = 2
Solution via A.solve(b):


<IPython.core.display.Math object>

### 6.2 無限多解（有自由變數）

$$
\begin{cases}
x + 2y + 3z = 6 \\
2x + 4y + 6z = 12
\end{cases}
$$

第二式是第一式的兩倍 → 本質上只有一個獨立方程。

In [9]:
A2 = sp.Matrix([
    [1, 2, 3],
    [2, 4, 6],
])
b2 = sp.Matrix([6, 12])
aug2 = A2.row_join(b2)

print("Augmented [A|b] =")
display_huge(aug2)

R2, pivots2 = aug2.rref()
print("RREF =")
display_huge(R2)

print(f"rank(A) = {A2.rank()}, rank([A|b]) = {aug2.rank()}, n = {A2.cols}")
print("Pivot columns (in A):", [p for p in pivots2 if p < A2.cols])
print("Free variables: y, z  (columns 1 and 2)")
print("From RREF: x + 2y + 3z = 6  =>  x = 6 - 2y - 3z")

Augmented [A|b] =


<IPython.core.display.Math object>

RREF =


<IPython.core.display.Math object>

rank(A) = 1, rank([A|b]) = 1, n = 3
Pivot columns (in A): [0]
Free variables: y, z  (columns 1 and 2)
From RREF: x + 2y + 3z = 6  =>  x = 6 - 2y - 3z


### 6.3 無解

$$
\begin{cases}
x + y = 1 \\
2x + 2y = 3
\end{cases}
$$

In [10]:
A3 = sp.Matrix([[1, 1], [2, 2]])
b3 = sp.Matrix([1, 3])
aug3 = A3.row_join(b3)

print("Augmented [A|b] =")
display_huge(aug3)

R3, pivots3 = aug3.rref()
print("RREF =")
display_huge(R3)

print(f"rank(A) = {A3.rank()}, rank([A|b]) = {aug3.rank()}")
print("Inconsistent: rank(A) < rank([A|b])  =>  no solution")

try:
    A3.solve(b3)
except Exception as e:
    print("A.solve(b) raises:", type(e).__name__, "—", e)

Augmented [A|b] =


<IPython.core.display.Math object>

RREF =


<IPython.core.display.Math object>

rank(A) = 1, rank([A|b]) = 2
Inconsistent: rank(A) < rank([A|b])  =>  no solution
A.solve(b) raises: NonInvertibleMatrixError — Matrix det == 0; not invertible.


## 7. 綜合練習：從 RREF 讀出參數解

$$
\begin{cases}
x + 2y + z = 4 \\
2x + 4y + 3z = 9 \\
3x + 6y + 4z = 13
\end{cases}
$$

請用增廣矩陣的 RREF 判斷解的情形，並寫出參數形式。

In [11]:
A4 = sp.Matrix([
    [1, 2, 1],
    [2, 4, 3],
    [3, 6, 4],
])
b4 = sp.Matrix([4, 9, 13])
aug4 = A4.row_join(b4)

print("Augmented [A|b] =")
display_huge(aug4)

R4, pivots4 = aug4.rref()
print("RREF =")
display_huge(R4)

print(f"rank(A) = {A4.rank()}, rank([A|b]) = {aug4.rank()}, n = {A4.cols}")
print("Pivot columns:", list(pivots4))

# Parametric solution: free variable y = t
t = sp.symbols("t")
# From RREF: x + 2y = 3,  z = 1  =>  x = 3 - 2t, y = t, z = 1
sol = sp.Matrix([3 - 2 * t, t, 1])
print("Parametric solution (y = t):")
display_huge(sol)

# Verify for a few t values
for val in [0, 1, -2]:
    x_val = sol.subs(t, val)
    print(f"t = {val}: A x = {A4 * x_val}, b = {b4}, match? {A4 * x_val == b4}")

Augmented [A|b] =


<IPython.core.display.Math object>

RREF =


<IPython.core.display.Math object>

rank(A) = 2, rank([A|b]) = 2, n = 3
Pivot columns: [0, 2]
Parametric solution (y = t):


<IPython.core.display.Math object>

t = 0: A x = Matrix([[4], [9], [13]]), b = Matrix([[4], [9], [13]]), match? True
t = 1: A x = Matrix([[4], [9], [13]]), b = Matrix([[4], [9], [13]]), match? True
t = -2: A x = Matrix([[4], [9], [13]]), b = Matrix([[4], [9], [13]]), match? True


## 小結

| 概念 | SymPy 方法 | 重點 |
|------|-----------|------|
| 列運算 | `row_swap` / `row_op` | 解不變的三種基本操作 |
| RREF | `A.rref()` | 回傳 `(RREF, pivots)`；RREF 唯一 |
| 主元 | `pivots` | 主元個數 = 秩 |
| 秩 | `A.rank()` | $0 \le \operatorname{rank}(A) \le \min(m,n)$ |
| 可逆判定 | `rank(A) == n` | 等價於 $\det(A) \neq 0$ |
| 增廣矩陣 | `A.row_join(b)` | 用來解 $A\mathbf{x}=\mathbf{b}$ |
| 解的情形 | 比較 $\operatorname{rank}(A)$ 與 $\operatorname{rank}([A\mid b])$ | 無解 / 唯一 / 無限多 |

**記住：** RREF 把矩陣化成最乾淨的階梯形；秩就是階梯上主元的個數，也是矩陣獨立資訊的度量。